# ANAL-003: LB 0.779 Diagnostic Analysis

CV AUC 0.9776 vs LB 0.779 gap decomposition.
5 analyses: per-species CV AUC, soundscape AUC, per-class breakdown, zero-shot, calibration.

In [ ]:
import subprocess, os, time, json, ast, random, warnings
from pathlib import Path
from collections import defaultdict
T0 = time.time()

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

FORCE_CPU = False
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=10)
    gpu_caps = [float(x.strip()) for x in r.stdout.strip().split('\n') if x.strip()]
    min_cap = min(gpu_caps) if gpu_caps else 99.0
    print(f'GPU caps: {gpu_caps}')
    if min_cap < 7.0:
        print(f'SM {min_cap} < 7.0 (P100) — using CPU')
        FORCE_CPU = True
except Exception:
    print('nvidia-smi failed')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

DEVICE = torch.device('cpu') if FORCE_CPU else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available() and not FORCE_CPU
if USE_AMP:
    try:
        from torch.amp import autocast
        AMP_DEVICE = 'cuda'
    except ImportError:
        from torch.cuda.amp import autocast
        AMP_DEVICE = None
print(f'[{time.time()-T0:.1f}s] PyTorch {torch.__version__}, Device: {DEVICE}')

In [ ]:
# ── Paths & Constants (identical to training) ──
IS_KAGGLE = Path('/kaggle/input').exists()

def find_data_dir():
    if not IS_KAGGLE: return Path('../data/raw')
    for p in Path('/kaggle/input').iterdir():
        if (p / 'train.csv').exists(): return p
        if p.is_dir():
            for sub in p.iterdir():
                if sub.is_dir() and (sub / 'train.csv').exists(): return sub
    raise FileNotFoundError('train.csv not found')

def find_model_dir():
    candidates = [
        Path('/kaggle/input/notebooks/montyeternity/birdclef2026-baseline-b0-training'),
        Path('/kaggle/input/birdclef2026-baseline-b0-training'),
    ]
    for c in candidates:
        if c.exists() and list(c.glob('best_fold*.pth')):
            return c
    if IS_KAGGLE:
        nb_dir = Path('/kaggle/input/notebooks')
        if nb_dir.exists():
            for pth in nb_dir.rglob('best_fold*.pth'):
                return pth.parent
    return Path('../models')

DATA_DIR = find_data_dir()
MODEL_DIR = find_model_dir()
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('../output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SR = 32_000; CLIP_DURATION = 5.0; CLIP_SAMPLES = int(SR * CLIP_DURATION)
N_FFT = 1024; HOP_LENGTH = 320; N_MELS = 128; FMIN = 50; FMAX = 14000
NUM_CLASSES = 234; SEED = 42; N_FOLDS = 5

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', nrows=0)
SPECIES_COLS = [c for c in submission.columns if c != 'row_id']
LABEL2IDX = {label: i for i, label in enumerate(SPECIES_COLS)}
IDX2LABEL = {i: label for label, i in LABEL2IDX.items()}
TAX_MAP = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name']))

print(f'[{time.time()-T0:.1f}s] Data: {DATA_DIR}')
print(f'Model: {MODEL_DIR} -> {list(MODEL_DIR.glob("best_fold*.pth"))}')
print(f'Samples: {len(train_df)}, Species: {len(SPECIES_COLS)}')

In [ ]:
# ── Audio + Model (identical to training) ──
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
AMP_TO_DB = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)
RESAMPLERS = {}

def load_audio(path, sr=SR, duration=CLIP_DURATION, offset=0.0):
    tl = int(sr * duration)
    try:
        waveform, file_sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if file_sr != sr:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, sr)
            waveform = RESAMPLERS[file_sr](waveform)
        start = int(offset * sr)
        waveform = waveform[:, start:start + tl].squeeze(0)
    except Exception:
        waveform = torch.zeros(tl)
    if waveform.numel() == 0:
        waveform = torch.zeros(tl)
    if waveform.numel() < tl:
        reps = (tl // waveform.numel()) + 1
        waveform = waveform.repeat(reps)[:tl]
    return waveform[:tl]

def audio_to_mel(waveform):
    mel = AMP_TO_DB(MEL_TRANSFORM(waveform.unsqueeze(0))).squeeze(0)
    return (mel - mel.mean()) / (mel.std() + 1e-6)

class BirdCLEFB0(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b0_ns', pretrained=False,
            in_chans=1, num_classes=0, global_pool='avg')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(1280, NUM_CLASSES)
    def forward(self, mel):
        return self.fc(self.dropout(self.backbone(mel)))

model = BirdCLEFB0().to(DEVICE)
wt_path = sorted(MODEL_DIR.glob('best_fold*.pth'))[0]
model.load_state_dict(torch.load(wt_path, map_location=DEVICE, weights_only=True))
model.eval()
print(f'[{time.time()-T0:.1f}s] Model loaded: {wt_path.name}')

In [ ]:
# ── Analysis A: Per-species CV AUC on validation set ──
print('=== Analysis A: Training Validation AUC ===')

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(train_df, train_df['primary_label'].astype(str), train_df['author'].astype(str)))
_, val_idx = folds[0]
val_df = train_df.iloc[val_idx].reset_index(drop=True)
print(f'Fold 0 val set: {len(val_df)} samples')

audio_dir = DATA_DIR / 'train_audio'
all_preds, all_targets = [], []

for i in tqdm(range(len(val_df)), desc='Val inference'):
    row = val_df.iloc[i]
    audio = load_audio(str(audio_dir / row['filename']))
    mel = audio_to_mel(audio).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        if USE_AMP:
            with autocast(AMP_DEVICE) if AMP_DEVICE else autocast():
                logits = model(mel)
        else:
            logits = model(mel)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

    target = np.zeros(NUM_CLASSES, dtype=np.float32)
    p = str(row['primary_label'])
    if p in LABEL2IDX: target[LABEL2IDX[p]] = 1.0
    sr_labels = row.get('secondary_labels', '[]')
    if isinstance(sr_labels, str) and sr_labels not in ('[]', ''):
        try:
            for s in ast.literal_eval(sr_labels):
                if str(s) in LABEL2IDX: target[LABEL2IDX[str(s)]] = 1.0
        except: pass

    all_preds.append(probs)
    all_targets.append(target)

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_targets_bin = (all_targets > 0.5).astype(np.float32)

per_species_cv_auc = {}
for i in range(NUM_CLASSES):
    col_targets = all_targets_bin[:, i]
    if 0 < col_targets.sum() < len(col_targets):
        try:
            auc = roc_auc_score(col_targets, all_preds[:, i])
        except:
            auc = None
    elif col_targets.sum() == 0:
        auc = None
    else:
        auc = None
    per_species_cv_auc[SPECIES_COLS[i]] = auc

valid_aucs = [v for v in per_species_cv_auc.values() if v is not None]
cv_macro_auc = np.mean(valid_aucs) if valid_aucs else 0.0
print(f'\n[{time.time()-T0:.1f}s] CV macro AUC (recomputed): {cv_macro_auc:.4f} ({len(valid_aucs)} species)')

worst_cv = sorted([(k, v) for k, v in per_species_cv_auc.items() if v is not None], key=lambda x: x[1])[:20]
print(f'\nWorst 20 species (CV):')
for sp, auc in worst_cv:
    cls = TAX_MAP.get(sp, '?')
    n_samples = len(train_df[train_df['primary_label'].astype(str) == sp])
    print(f'  {sp:15s} ({cls:12s}) AUC={auc:.4f} n={n_samples}')

In [ ]:
# ── Analysis B: Soundscape Validation AUC (key metric) ──
print('\n=== Analysis B: Soundscape AUC ===')

sc_labels_path = DATA_DIR / 'train_soundscapes_labels.csv'
sc_dir = DATA_DIR / 'train_soundscapes'

if not sc_labels_path.exists():
    print('ERROR: train_soundscapes_labels.csv not found')
    sc_macro_auc = None
    per_species_sc_auc = {}
else:
    sc_df = pd.read_csv(sc_labels_path)
    print(f'Soundscape labels: {len(sc_df)} segments')
    print(f'Columns: {list(sc_df.columns)}')
    print(sc_df.head(3))

    sc_species_in_labels = set()
    for _, row in sc_df.iterrows():
        for lbl in str(row.get('primary_label', '')).split(';'):
            lbl = lbl.strip()
            if lbl and lbl in LABEL2IDX:
                sc_species_in_labels.add(lbl)
    print(f'Species in soundscape labels: {len(sc_species_in_labels)}')

    sc_preds, sc_targets = [], []
    sc_audio_cache = {}

    for idx in tqdm(range(len(sc_df)), desc='Soundscape inference'):
        row = sc_df.iloc[idx]
        filename = str(row.get('filename', row.get('audio_id', '')))
        if not filename.endswith('.ogg'):
            filename = filename + '.ogg'

        start_sec = float(row.get('start_time', row.get('seconds', 0)))
        end_sec = float(row.get('end_time', start_sec + 5))

        filepath = sc_dir / filename
        if not filepath.exists():
            for f in sc_dir.iterdir():
                if filename.replace('.ogg','') in f.name:
                    filepath = f
                    break

        cache_key = str(filepath)
        if cache_key not in sc_audio_cache:
            try:
                wav, fsr = torchaudio.load(str(filepath))
                if wav.shape[0] > 1:
                    wav = wav.mean(dim=0, keepdim=True)
                if fsr != SR:
                    if fsr not in RESAMPLERS:
                        RESAMPLERS[fsr] = torchaudio.transforms.Resample(fsr, SR)
                    wav = RESAMPLERS[fsr](wav)
                sc_audio_cache[cache_key] = wav.squeeze(0)
            except Exception as e:
                if idx < 3: print(f'  WARN: {filepath.name}: {e}')
                sc_audio_cache[cache_key] = torch.zeros(SR * 60)

        full_wav = sc_audio_cache[cache_key]
        s0 = int(start_sec * SR)
        clip = full_wav[s0:s0 + CLIP_SAMPLES]
        if clip.numel() < CLIP_SAMPLES:
            clip = F.pad(clip, (0, CLIP_SAMPLES - clip.numel()))

        mel = audio_to_mel(clip).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            if USE_AMP:
                with autocast(AMP_DEVICE) if AMP_DEVICE else autocast():
                    logits = model(mel)
            else:
                logits = model(mel)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        for lbl in str(row.get('primary_label', '')).split(';'):
            lbl = lbl.strip()
            if lbl in LABEL2IDX:
                target[LABEL2IDX[lbl]] = 1.0

        sc_preds.append(probs)
        sc_targets.append(target)

    sc_preds = np.array(sc_preds)
    sc_targets = np.array(sc_targets)
    sc_targets_bin = (sc_targets > 0.5).astype(np.float32)

    per_species_sc_auc = {}
    for i in range(NUM_CLASSES):
        col = sc_targets_bin[:, i]
        if 0 < col.sum() < len(col):
            try:
                per_species_sc_auc[SPECIES_COLS[i]] = roc_auc_score(col, sc_preds[:, i])
            except:
                per_species_sc_auc[SPECIES_COLS[i]] = None
        else:
            per_species_sc_auc[SPECIES_COLS[i]] = None

    valid_sc_aucs = [v for v in per_species_sc_auc.values() if v is not None]
    sc_macro_auc = np.mean(valid_sc_aucs) if valid_sc_aucs else 0.0
    print(f'\n[{time.time()-T0:.1f}s] Soundscape macro AUC: {sc_macro_auc:.4f} ({len(valid_sc_aucs)} species)')
    print(f'Domain shift gap: CV {cv_macro_auc:.4f} - SC {sc_macro_auc:.4f} = {cv_macro_auc - sc_macro_auc:.4f}')

    worst_sc = sorted([(k, v) for k, v in per_species_sc_auc.items() if v is not None], key=lambda x: x[1])[:20]
    print(f'\nWorst 20 species (Soundscape):')
    for sp, auc in worst_sc:
        cls = TAX_MAP.get(sp, '?')
        cv = per_species_cv_auc.get(sp)
        cv_str = f'{cv:.3f}' if cv is not None else 'N/A'
        print(f'  {sp:15s} ({cls:12s}) SC_AUC={auc:.4f} CV_AUC={cv_str}')

In [ ]:
# ── Analysis C: Per-class (Aves/Amphibia/Insecta/Mammalia) breakdown ──
print('\n=== Analysis C: Per-class AUC ===')

class_cv_aucs = defaultdict(list)
class_sc_aucs = defaultdict(list)

for sp in SPECIES_COLS:
    cls = TAX_MAP.get(sp, 'Unknown')
    cv = per_species_cv_auc.get(sp)
    sc = per_species_sc_auc.get(sp)
    if cv is not None: class_cv_aucs[cls].append(cv)
    if sc is not None: class_sc_aucs[cls].append(sc)

class_results = {}
print(f'{"Class":12s} {"CV AUC":>10s} {"SC AUC":>10s} {"Gap":>8s} {"CV_n":>5s} {"SC_n":>5s}')
print('-' * 55)
for cls in ['Aves', 'Amphibia', 'Insecta', 'Mammalia', 'Reptilia']:
    cv_vals = class_cv_aucs.get(cls, [])
    sc_vals = class_sc_aucs.get(cls, [])
    cv_mean = np.mean(cv_vals) if cv_vals else None
    sc_mean = np.mean(sc_vals) if sc_vals else None
    gap = (cv_mean - sc_mean) if (cv_mean and sc_mean) else None
    class_results[cls] = {'cv_auc': cv_mean, 'sc_auc': sc_mean, 'gap': gap,
                          'cv_n': len(cv_vals), 'sc_n': len(sc_vals)}
    cv_s = f'{cv_mean:.4f}' if cv_mean else 'N/A'
    sc_s = f'{sc_mean:.4f}' if sc_mean else 'N/A'
    gap_s = f'{gap:.4f}' if gap else 'N/A'
    print(f'{cls:12s} {cv_s:>10s} {sc_s:>10s} {gap_s:>8s} {len(cv_vals):>5d} {len(sc_vals):>5d}')

In [ ]:
# ── Analysis D: Zero-shot species ──
print('\n=== Analysis D: Zero-shot Species ===')

train_species = set(train_df['primary_label'].astype(str).unique())
all_species = set(SPECIES_COLS)
zero_shot = all_species - train_species
print(f'Zero-shot species: {len(zero_shot)}')

zs_results = []
for sp in sorted(zero_shot):
    idx = LABEL2IDX[sp]
    cls = TAX_MAP.get(sp, '?')

    sc_auc = per_species_sc_auc.get(sp)
    sc_preds_col = sc_preds[:, idx] if sc_preds is not None else np.array([])
    sc_target_col = sc_targets_bin[:, idx] if sc_targets_bin is not None else np.array([])

    n_positive = int(sc_target_col.sum()) if len(sc_target_col) > 0 else 0
    pred_at_pos = sc_preds_col[sc_target_col > 0.5] if n_positive > 0 else np.array([])
    pred_at_neg = sc_preds_col[sc_target_col < 0.5]

    res = {
        'species': sp, 'class': cls,
        'sc_n_positive': n_positive,
        'sc_auc': sc_auc,
        'pred_at_pos_mean': float(pred_at_pos.mean()) if len(pred_at_pos) > 0 else None,
        'pred_at_pos_max': float(pred_at_pos.max()) if len(pred_at_pos) > 0 else None,
        'pred_at_neg_mean': float(pred_at_neg.mean()) if len(pred_at_neg) > 0 else None,
    }
    zs_results.append(res)
    auc_s = f'{sc_auc:.3f}' if sc_auc is not None else 'N/A'
    pos_mean = f'{res["pred_at_pos_mean"]:.4f}' if res['pred_at_pos_mean'] is not None else 'N/A'
    pos_max = f'{res["pred_at_pos_max"]:.4f}' if res['pred_at_pos_max'] is not None else 'N/A'
    print(f'  {sp:15s} ({cls:10s}) n_pos={n_positive:>4d} SC_AUC={auc_s:>7s} pred@pos_mean={pos_mean} pred@pos_max={pos_max}')

In [ ]:
# ── Analysis E: Prediction calibration ──
print('\n=== Analysis E: Prediction Calibration ===')

if sc_targets_bin is not None and sc_preds is not None:
    pos_preds = sc_preds[sc_targets_bin > 0.5]
    neg_preds = sc_preds[sc_targets_bin < 0.5]

    print(f'Positive predictions (n={len(pos_preds)}):')
    print(f'  mean={pos_preds.mean():.4f} std={pos_preds.std():.4f} median={np.median(pos_preds):.4f}')
    print(f'  min={pos_preds.min():.6f} max={pos_preds.max():.6f}')

    print(f'Negative predictions (n={len(neg_preds)}):')
    print(f'  mean={neg_preds.mean():.6f} std={neg_preds.std():.6f} median={np.median(neg_preds):.6f}')

    print(f'\nSeparation ratio: {pos_preds.mean() / (neg_preds.mean() + 1e-8):.1f}x')

    print(f'\nPositive prediction distribution:')
    for t in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
        frac = (pos_preds > t).mean()
        print(f'  > {t:.2f}: {frac:.1%}')

    print(f'\nThreshold analysis (on soundscape):')
    for threshold in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]:
        pred_binary = (sc_preds > threshold).astype(float)
        tp = ((pred_binary > 0.5) & (sc_targets_bin > 0.5)).sum()
        fp = ((pred_binary > 0.5) & (sc_targets_bin < 0.5)).sum()
        fn = ((pred_binary < 0.5) & (sc_targets_bin > 0.5)).sum()
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        print(f'  t={threshold:.2f}: P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={int(tp)} FP={int(fp)} FN={int(fn)}')

print(f'\n[{time.time()-T0:.1f}s] Analysis E done')

In [ ]:
# ── Save diagnostic results ──
results = {
    'cv_macro_auc': float(cv_macro_auc),
    'sc_macro_auc': float(sc_macro_auc) if sc_macro_auc is not None else None,
    'domain_shift_gap': float(cv_macro_auc - sc_macro_auc) if sc_macro_auc else None,
    'n_species_cv': len([v for v in per_species_cv_auc.values() if v is not None]),
    'n_species_sc': len([v for v in per_species_sc_auc.values() if v is not None]),
    'per_species_cv_auc': {k: float(v) if v is not None else None for k, v in per_species_cv_auc.items()},
    'per_species_sc_auc': {k: float(v) if v is not None else None for k, v in per_species_sc_auc.items()},
    'class_results': {k: {kk: float(vv) if vv is not None else None for kk, vv in v.items()} for k, v in class_results.items()},
    'zero_shot_results': zs_results,
    'calibration': {
        'pos_pred_mean': float(pos_preds.mean()) if len(pos_preds) > 0 else None,
        'neg_pred_mean': float(neg_preds.mean()) if len(neg_preds) > 0 else None,
        'separation_ratio': float(pos_preds.mean() / (neg_preds.mean() + 1e-8)) if len(pos_preds) > 0 else None,
    },
    'total_time_seconds': time.time() - T0,
}

out_path = OUTPUT_DIR / 'diagnostic_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {out_path} ({out_path.stat().st_size/1024:.1f}KB)')
print(f'Total analysis time: {time.time()-T0:.1f}s')

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
print(f'CV macro AUC:         {cv_macro_auc:.4f}')
print(f'Soundscape macro AUC: {sc_macro_auc:.4f}' if sc_macro_auc else 'Soundscape: N/A')
if sc_macro_auc:
    print(f'Domain shift gap:     {cv_macro_auc - sc_macro_auc:.4f}')
print(f'Zero-shot species:    {len(zero_shot)}')
print(f'LB Score:             0.779')